In [10]:
import pdfplumber
import pandas as pd
import os
import re
from img2table.document import Image
from io import BytesIO
from pdf2image import convert_from_path
from img2table.ocr import TesseractOCR
from img2table.document import PDF
import json

# Functions and tool setup

In [11]:
ocr = TesseractOCR()

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.58 : libtiff 4.7.2 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.8 zlib/1.2.12 liblzma/5.8.3 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.3 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.64.0


In [12]:
def extract_headers(page):
    lines = page.extract_text_lines()
    return pd.DataFrame([line for line in lines if re.search('^[A-Z]{1}[0-9]{1,2}', line['text']) is not None])

In [13]:
def select_best_heading(page, table, headers):
    if len(headers) == 0:
        return ''
    headers['is_above'] = headers['top'].apply(lambda x: x < table.bbox.relative.y1 * page.height)
    headers = headers[headers['is_above']].sort_values('top', ascending=False).reset_index(drop=True)
    if len(headers) == 0:
        return ''
    return headers['text'][0]

In [14]:
def parse_page_tables(file, page_index):
    pdf = pdfplumber.open(file)
    page = pdf.pages[page_index]
    headers = extract_headers(page)

    doc = PDF(file,
          pages=[page_index],
          detect_rotation=False,
          pdf_text_extraction=True)

    results = doc.extract_tables(ocr=ocr,
                    implicit_rows=False,
                    implicit_columns=False,
                    borderless_tables=False,
                    min_confidence=50,
                    max_workers=1)

    data = []
    for table in results[page_index]:
        heading = select_best_heading(page, table, headers)
        code_match = re.search('^[A-Z]{1}[0-9]+', heading)
        if code_match is not None:
            code = code_match.group(0).strip()
        else:
            code = ''

        heading = heading.replace(code, '')
        heading = re.sub('^\\.', '', heading)
        heading = heading.strip()

        data.append({
            'table_code': code,
            'table_title': heading,
            'table_records': table.df.fillna('').to_dict(orient='records'),
            'table_html': re.sub('[\n ]+', ' ', table.html)
        })

    return data

In [15]:
def extract_data_from_file(file):
    year = file.split('/')[-2].strip()
    print(file)
    try:
        pdf = pdfplumber.open(file)
    except:
        return None
    unitid = file.split('/')[-1].replace('.pdf', '').strip()

    data = []
    for page in range(0, len(pdf.pages)):
        try:
            data.extend(parse_page_tables(file, page))
        except:
            continue

    if len(data) == 0:
        return None

    data = pd.DataFrame(data)

    data = data[data['table_code'] != '']

    data = data.groupby('table_code').agg({
        'table_code': first,
        'table_title': first,
        'table_records': list,
        'table_html': lambda tables: '<div class="separator"></div>'.join(tables)
    }).reset_index(drop=True)

    data['table_num'] = data['table_code'].apply(lambda x: re.search('[0-9]+', x).group(0)).apply(int)
    data['section'] = data['table_code'].apply(lambda x: re.search('[A-Z]+', x).group(0))
    data = data.sort_values(['section', 'table_num'])

    data = data[['section', 'table_num', 'table_code', 'table_title', 'table_records', 'table_html']]
    data['unitid'] = unitid
    data['file'] = file
    data['year'] = year

    return data

In [16]:
def first(lst):
    return list(lst)[0]

In [17]:
directory = pd.read_csv('../data/ipeds/hd2025.csv')
directory = directory[['UNITID', 'INSTNM', 'STABBR']]

# Parsing files

In [66]:
years = pd.DataFrame({
    'year': [year for year in os.listdir('../data/cds-docs') if 'DS_Store' not in year and 'no-date' not in year]
})

In [67]:
years['start_year'] = years['year'].apply(lambda x: x.split('-')[0]).apply(int)
years['end_year'] = years['year'].apply(lambda x: x.split('-')[1]).apply(int)

In [68]:
years = years.query('start_year >= 2021 and end_year < 2026')
years = years.sort_values('start_year', ascending=False)
years = years['year']

In [69]:
all_files = []
for year in years:
    all_files.extend([f'../data/cds-docs/{year}/{file}' for file in os.listdir(f'../data/cds-docs/{year}') if '.DS_Store' not in file])

In [70]:
all_files = pd.DataFrame({
    'path': all_files
})

In [71]:
all_files['unitid'] = all_files['path'].apply(lambda x: x.split('/')[-1].replace('.pdf', '').strip())
all_files['year'] = all_files['path'].apply(lambda x: x.split('/')[-2])

In [72]:
all_files = all_files.groupby('unitid').agg({
    'path': list,
    'year': lambda x: list(x)[0]
}).reset_index()

In [73]:
# for i in range(0, len(all_files)):
#     unitid = all_files['unitid'][i]
#     out_path = f'../data/raw-parse/{unitid}.csv'

#     if i % 20 == 0:
#         print(f'{i / len(all_files)*100}% complete')

#     if os.path.exists(out_path):
#         print(f'Already scraped {unitid}')
#         continue
#     else:
#         try:
#             df = pd.concat([extract_data_from_file(file) for file in all_files['path'][i]])
#         except Exception as e:
#             print(f'{unitid}, {e}')
#             continue
#         df.to_csv(out_path, index=False)

In [160]:
site_directory = pd.read_csv('../data/file-inventory.csv')
site_directory['file'] = 'https://media.githubusercontent.com/media/declanrjb/cds-engine/refs/heads/main/' + site_directory['file']
site_directory['year'] = site_directory['year'].apply(lambda year: f'{year}-{year+1}')
site_directory['unitid'] = site_directory['unitid'].apply(str)

In [161]:
df = pd.concat([pd.read_csv(f'../data/raw-parse/{file}') for file in os.listdir('../data/raw-parse')])

In [162]:
df['file'] = df['file'].apply(lambda x: x.replace('../', 'https://media.githubusercontent.com/media/declanrjb/cds-engine/refs/heads/main/'))

In [163]:
df['file'] = df['file'].apply(lambda x: x.replace('../', ''))

In [164]:
df['year_num'] = df['year'].apply(lambda x: int(x.split('-')[0]))
df = df.sort_values('year', ascending=False).reset_index(drop=True)

In [165]:
df['unitid'] = df['unitid'].apply(str)
directory['UNITID'] = directory['UNITID'].apply(str)

In [166]:
df['unitid'] = df['unitid'].apply(lambda x: x.split('_')[0])

In [167]:
data = {}

In [ ]:
for unitid in site_directory['unitid'].unique():
    temp_directory = directory.query(f'UNITID == "{unitid}"')
    temp_directory['UNITID'] = temp_directory['UNITID'].apply(int)

    college_df = df.query(f'unitid == "{unitid}"').reset_index(drop=True)
    site_files = site_directory.query(f'unitid == "{unitid}"')

    if len(temp_directory) == 0:
        print(f'{unitid} has no ipeds entry')
        continue
        
    inst_data = temp_directory.to_dict(orient='records')[0]
    inst_data['years'] = {}

    for year, file in zip(site_files['year'], site_files['file']):
        inst_data['years'][year] = {}
        inst_data['years'][year]['file'] = file

        temp_tables = college_df.query(f'year == "{year}"').reset_index(drop=True)
        if len(temp_tables) > 0:
            inst_data['years'][year]['parsed_tables'] = temp.to_dict(orient='records')
        else:
            print(f'{unitid}, {year} has no parsed files')

    with open(f'../web-app/colleges/{unitid}.json', 'w') as out_file:
        json.dump(inst_data, out_file)

KeyError: 'unitid'

In [ ]:
with open('../web-app/data.json', 'w') as out_file:
    json.dump(data, out_file)